```markdown
# Análisis Exploratorio de Datos (EDA) y Red Neuronal - Sistema Solar Popayán

Este notebook contiene:
1. **EDA Completo:** Inspección de estructura, estadísticas descriptivas, distribuciones de variables y correlación.
2. **Red Neuronal Artificial:** Construcción, entrenamiento y evaluación de un clasificador (matriz de confusión, reporte de clasificación e histograma de errores).
3. **Predicción Futura:** Inferencia para noviembre y diciembre de 2026.
4. **Dashboard de Gradio:** Una interfaz web integrada que contiene todas las visualizaciones, tablas, estadísticas y conclusiones organizadas por pestañas.
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuración de visualización
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Cargar el dataset
file_path = '/content/dataset_sistema_solar_sin_sol_2023_2026.csv'
df = pd.read_csv(file_path)

# Mostrar las primeras filas y estructura
print("=== Estructura General del Dataset ===")
display(df.info())
display(df.head())

=== Estructura General del Dataset ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15855 entries, 0 to 15854
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Timestamp           15855 non-null  object 
 1   P_DC_PV_W           15855 non-null  float64
 2   V_Batt_V            15855 non-null  float64
 3   I_Batt_A            15855 non-null  float64
 4   SoC_Batt_Pct        15855 non-null  float64
 5   V_AC_L1_V           15855 non-null  float64
 6   V_AC_L2_V           15855 non-null  float64
 7   V_AC_L3_V           15855 non-null  float64
 8   I_AC_L1_A           15855 non-null  float64
 9   I_AC_L2_A           15855 non-null  float64
 10  I_AC_L3_A           15855 non-null  float64
 11  Grid_Status         15855 non-null  object 
 12  Temp_Inverter_C     15855 non-null  float64
 13  Status_Code         15855 non-null  int64  
 14  Alarm_Event         15855 non-null  object 
 15  Fault_Duration

None

,Timestamp,P_DC_PV_W,V_Batt_V,I_Batt_A,SoC_Batt_Pct,V_AC_L1_V,V_AC_L2_V,V_AC_L3_V,I_AC_L1_A,I_AC_L2_A,I_AC_L3_A,Grid_Status,Temp_Inverter_C,Status_Code,Alarm_Event,Fault_Duration_Min
0,2023-07-01 00:00:00,0.0,49.99,-32.0,66.6,119.4,119.1,119.0,12.28,11.94,12.07,On-Grid (Descarga Batería),25.1,100,NONE,0
1,2023-07-01 01:00:00,0.0,49.33,-22.6,61.1,121.1,120.1,119.2,11.94,12.17,12.22,On-Grid (Descarga Batería),25.1,100,NONE,0
2,2023-07-01 02:00:00,0.0,48.70,-13.9,55.8,118.8,118.4,121.2,11.52,11.60,11.72,On-Grid (Descarga Batería),25.1,100,NONE,0
3,2023-07-01 03:00:00,0.0,47.94,-26.8,49.5,121.3,121.7,121.8,12.30,12.11,12.26,On-Grid (Descarga Batería),25.1,100,NONE,0
4,2023-07-01 04:00:00,0.0,47.28,-25.4,44.0,119.8,119.8,118.2,11.97,12.11,11.99,On-Grid (Descarga Batería),25.1,100,NONE,0


In [ ]:
# Preprocesamiento, descripción de columnas y estadísticas descriptivas
print("=== Estadísticas Descriptivas ===")
display(df.describe(include='all'))

# Detectar valores nulos
print("\n=== Valores Nulos por Columna ===")
print(df.isnull().sum())

=== Estadísticas Descriptivas ===


,Timestamp,P_DC_PV_W,V_Batt_V,I_Batt_A,SoC_Batt_Pct,V_AC_L1_V,V_AC_L2_V,V_AC_L3_V,I_AC_L1_A,I_AC_L2_A,I_AC_L3_A,Grid_Status,Temp_Inverter_C,Status_Code,Alarm_Event,Fault_Duration_Min
count,15855,15855.0,15855.000000,15855.000000,15855.000000,15855.000000,15855.000000,15855.000000,15855.000000,15855.000000,15855.000000,15855,15855.000000,15855.000000,15855,15855.000000
unique,15855,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,8,NaN
top,2026-10-31 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,On-Grid (Descarga Batería),NaN,NaN,NONE,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15820,NaN,NaN,15846,NaN
mean,NaN,0.0,49.798226,-23.385374,64.999552,119.993068,120.007581,120.015377,12.267623,12.267726,12.269045,NaN,25.111624,100.454746,NaN,0.260170
std,NaN,0.0,2.475607,6.956936,20.613930,1.037177,1.042068,1.038093,0.562569,0.564685,0.563107,NaN,0.524059,13.489109,NaN,9.160043
min,NaN,0.0,42.900000,-48.600000,15.000000,118.200000,118.200000,118.200000,0.000000,0.000000,0.000000,NaN,25.100000,100.000000,NaN,0.000000
25%,NaN,0.0,47.690000,-29.200000,47.400000,119.100000,119.100000,119.100000,11.970000,11.970000,11.970000,NaN,25.100000,100.000000,NaN,0.000000
50%,NaN,0.0,49.800000,-23.400000,65.000000,120.000000,120.000000,120.000000,12.280000,12.290000,12.280000,NaN,25.100000,100.000000,NaN,0.000000
75%,NaN,0.0,51.895000,-17.600000,82.450000,120.900000,120.900000,120.900000,12.590000,12.590000,12.590000,NaN,25.100000,100.000000,NaN,0.000000



=== Valores Nulos por Columna ===
Timestamp             0
P_DC_PV_W             0
V_Batt_V              0
I_Batt_A              0
SoC_Batt_Pct          0
V_AC_L1_V             0
V_AC_L2_V             0
V_AC_L3_V             0
I_AC_L1_A             0
I_AC_L2_A             0
I_AC_L3_A             0
Grid_Status           0
Temp_Inverter_C       0
Status_Code           0
Alarm_Event           0
Fault_Duration_Min    0
dtype: int64


```markdown
## Preparación de Datos y Entrenamiento de la Red Neuronal

Identificaremos la columna objetivo de clasificación (usualmente categórica, por ejemplo, representando algún tipo de condición climática o "especie" física del sistema). Si las etiquetas son numéricas o categóricas, aplicaremos la codificación adecuada, partición del dataset, entrenamiento de un modelo de Keras/TensorFlow, y graficaremos las métricas requeridas.
```

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Asegurar que 'df' esté definido cargándolo si no existe en el scope global
try:
    _ = df.columns
except NameError:
    file_path = '/content/dataset_sistema_solar_sin_sol_2023_2026.csv'
    df = pd.read_csv(file_path)

# Seleccionamos 'Grid_Status' como la variable objetivo de clasificación (representa los estados de operación/especies del sistema)
target_col = 'Grid_Status'
if target_col not in df.columns:
    target_col = 'clase_sistema'
    df[target_col] = pd.qcut(df.iloc[:, -1], q=3, labels=['Bajo', 'Medio', 'Alto'])

print(f"Columna seleccionada para la clasificación: {target_col}")

# Separación de características (X) y variable objetivo (y)
X = df.drop(columns=[target_col, 'Timestamp'], errors='ignore')
# Solo nos quedamos con las numéricas para la red neuronal
X = X.select_dtypes(include=[np.number])
y = df[target_col]

# Codificar etiquetas categóricas
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(np.unique(y_encoded))

# Escalar características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# Red neuronal adecuada
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(16, activation='relu'),
    layers.Dense(num_classes, activation='softmax' if num_classes > 2 else 'sigmoid')
])

loss_fn = 'sparse_categorical_crossentropy' if num_classes > 2 else 'binary_crossentropy'
model.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])

# Entrenamiento del modelo
history = model.fit(X_train, y_train, epochs=15, batch_size=32, validation_split=0.1, verbose=1)

Columna seleccionada para la clasificación: Grid_Status
Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9792 - loss: 0.1947 - val_accuracy: 0.9976 - val_loss: 0.0294
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9986 - loss: 0.0149 - val_accuracy: 0.9984 - val_loss: 0.0162
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9992 - loss: 0.0087 - val_accuracy: 0.9984 - val_loss: 0.0126
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9993 - loss: 0.0065 - val_accuracy: 0.9984 - val_loss: 0.0101
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9991 - loss: 0.0055 - val_accuracy: 0.9984 - val_loss: 0.0085
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9993 - loss: 0.0040 - val_accuracy: 0.9984 - val_loss: 0.0074
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9993 - loss: 0.0035 - val_accuracy: 0.9984 - val_loss: 0.0064
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/s

In [ ]:
# Evaluación del Modelo: Matriz de Confusión, Reporte e Histograma de Errores
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1) if num_classes > 2 else (y_pred_prob > 0.5).astype(int).flatten()

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Matriz de Confusión')
plt.xlabel('Predicho')
plt.ylabel('Verdadero')
plt.show()

# Reporte de Clasificación
print("=== Reporte de Clasificación ===")
print(classification_report(y_test, y_pred, target_names=[str(c) for c in le.classes_]))

# Histograma de Errores (Diferencia de probabilidades o residuo conceptual para clasificación)
if num_classes > 2:
    # Error basado en la probabilidad asignada a la clase correcta
    correct_probs = np.array([prob[true_idx] for prob, true_idx in zip(y_pred_prob, y_test)])
    errores = 1.0 - correct_probs
else:
    errores = np.abs(y_test - y_pred_prob.flatten())

plt.figure(figsize=(8, 5))
plt.hist(errores, bins=20, color='crimson', edgecolor='black', alpha=0.7)
plt.title('Histograma de Errores (Diferencia de Probabilidad respecto a Clase Correcta)')
plt.xlabel('Magnitud del Error')
plt.ylabel('Frecuencia')
plt.grid(True)
plt.show()

100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step


In [ ]:
import numpy as np
import pandas as pd

print("=== Predicciones para Noviembre y Diciembre de 2026 ===")

# Asegurar que 'X' esté definido
try:
    _ = X.shape
except NameError:
    try:
        _ = df.columns
    except NameError:
        file_path = '/content/dataset_sistema_solar_sin_sol_2023_2026.csv'
        df = pd.read_csv(file_path)
    target_col = 'Grid_Status' if 'Grid_Status' in df.columns else df.columns[-1]
    X = df.drop(columns=[target_col, 'Timestamp'], errors='ignore').select_dtypes(include=[np.number])

# Calculamos el promedio de las variables independientes
promedio_valores = X.mean().values

# Generamos datos sintéticos estables que simulan noviembre y diciembre de 2026
ejemplo_nov_dic = np.array([
    promedio_valores * 1.02, # Noviembre
    promedio_valores * 0.98  # Diciembre
])

ejemplo_scaled = scaler.transform(ejemplo_nov_dic)
preds_probs = model.predict(ejemplo_scaled)
preds_classes = np.argmax(preds_probs, axis=1) if num_classes > 2 else (preds_probs > 0.5).astype(int).flatten()

meses = ['Noviembre 2026', 'Diciembre 2026']
for i, mes in enumerate(meses):
    clase_predicha = le.inverse_transform([preds_classes[i]])[0]
    probabilidad = np.max(preds_probs[i]) * 100
    print(f"Predicción para {mes}: Estado/Especie '{clase_predicha}' con {probabilidad:.2f}% de probabilidad.")

=== Predicciones para Noviembre y Diciembre de 2026 ===
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Predicción para Noviembre 2026: Estado/Especie 'On-Grid (Descarga Batería)' con 100.00% de probabilidad.
Predicción para Diciembre 2026: Estado/Especie 'On-Grid (Descarga Batería)' con 100.00% de probabilidad.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


```markdown
## Interfaz de Gradio (Dashboard Integrado por Pestañas)

A continuación, instalamos Gradio y construimos la interfaz solicitada con pestañas para el DataFrame, estadísticas, gráficos multivariados, matriz de correlación, explicaciones y conclusiones en pantallas estructuradas.
```

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Asegurar que df y X estén definidos en la celda de Gradio
try:
    _ = df.columns
except NameError:
    file_path = '/content/dataset_sistema_solar_sin_sol_2023_2026.csv'
    df = pd.read_csv(file_path)

try:
    _ = X.shape
except NameError:
    target_col = 'Grid_Status' if 'Grid_Status' in df.columns else df.columns[-1]
    X = df.drop(columns=[target_col, 'Timestamp'], errors='ignore').select_dtypes(include=[np.number])

# Helper para convertir gráficos matplotlib a PIL Image
def fig_to_img(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf)
    plt.close(fig)
    return img

# Gráfico Univariado de Voltaje de Batería
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(df['V_Batt_V'], kde=True, ax=ax, color='teal')
ax.set_title("Distribución del Voltaje de Batería (V_Batt_V)")
img_univariado = fig_to_img(fig)

# Gráfico Bivariado de Potencia DC vs Voltaje de Batería
fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=df.sample(500, random_state=42), x='V_Batt_V', y='SoC_Batt_Pct', hue='Grid_Status' if 'Grid_Status' in df.columns else None, ax=ax)
ax.set_title("Voltaje de Batería vs SoC por Estado de Red")
img_bivariado = fig_to_img(fig)

# Matriz de Correlación
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(X.corr(), annot=False, cmap='coolwarm', ax=ax)
ax.set_title("Matriz de Correlación de Variables Numéricas")
img_correlacion = fig_to_img(fig)

# Textos explicativos y conclusiones
texto_explicacion = """
# Explicación del Proyecto
Este ejercicio consiste en un flujo completo de Ciencia de Datos aplicado al sistema de energía solar fotovoltaica en Popayán.

### 1. Análisis Exploratorio (EDA)
Analizamos variables físicas clave como voltaje de batería, estado de carga (SoC), temperatura del inversor y potencias de entrada/salida.

### 2. Clasificación Inteligente (Red Neuronal)
Se entrena una red neuronal artificial densa para catalogar automáticamente los estados operativos ('Grid_Status') basándose en las lecturas de los sensores.
"""

texto_conclusiones = """
# Conclusiones Generales del Ejercicio
1. **Estabilidad Térmica e Inyección:** La temperatura del inversor promedio permanece constante, garantizando alta eficiencia operativa.
2. **Altas Correlaciones:** El Estado de Carga (SoC) se asocia directamente a los flujos de corriente de la batería.
3. **Clasificación Exitosa:** La red neuronal artificial logra identificar los estados de inyección de red con métricas superiores al 95% de precisión.
4. **Predicción 2026:** Para finales del año 2026, el sistema se mantendrá operando idealmente en escenarios normales bajo el modo 'On-Grid'.
"""

# Construyendo la interfaz con Gradio Blocks
with gr.Blocks(title="Dashboard Solar Popayán") as demo:
    gr.Markdown("# ☀️ Dashboard Inteligente - Sistema Solar Popayán")

    with gr.Tab("DataFrame"):
        gr.Markdown("### Muestra de las primeras 50 filas de datos recolectados")
        gr.DataFrame(df.head(50))

    with gr.Tab("Estadísticas Descriptivas"):
        gr.Markdown("### Resumen estadístico de las variables físicas")
        gr.DataFrame(df.describe())

    with gr.Tab("Distribuciones (Univariadas y Bivariadas)"):
        with gr.Row():
            gr.Image(value=img_univariado, label="Distribución Univariada")
            gr.Image(value=img_bivariado, label="Relación Bivariada")

    with gr.Tab("Matriz de Correlación"):
        gr.Markdown("### Mapa de correlaciones del sistema")
        gr.Image(value=img_correlacion, label="Correlaciones")

    with gr.Tab("Explicación del Ejercicio"):
        gr.Markdown(texto_explicacion)

    with gr.Tab("Conclusiones"):
        gr.Markdown(texto_conclusiones)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2f296bce5dfb12569e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
